# 检索与 RAG：召回、混合排序、证据与拒答

从数据入库到答案验真解释完整 RAG 链路，手算融合和召回指标，定位检索与生成各自的失败。

Python 3.10+ / CPU / 无密钥。固定教学 fixture，不调用模型或服务。先读对应正文，预测成功与失败，再运行并修改一个边界。

In [ ]:
def rrf(lists, k=60):
    if k <= 0: raise ValueError('k')
    scores={}
    for ranking in lists:
        if len(set(ranking)) != len(ranking): raise ValueError('duplicate ID')
        for rank, doc in enumerate(ranking, 1):
            scores[doc]=scores.get(doc,0)+1/(k+rank)
    return sorted(scores, key=lambda d:(-scores[d],d))

ranked=rrf([['a','b','c'],['b','c','d']])
assert ranked == ['b','c','a','d']
gold={'b','d'}
recall2=len(set(ranked[:2]) & gold)/len(gold)
assert recall2 == 0.5
# 权限先过滤，两条召回都只能看到 a/b。
allowed={'a','b'}
safe=rrf([[d for d in r if d in allowed]
          for r in [['a','b','c'],['b','c','d']]])
assert safe == ['b','a'] and not set(safe)-allowed
checks={'rank_fusion':True,'recall_at_k':True,'acl_before_context':True}
print(ranked,recall2,safe)

## 排名与标签口径
指数gain、log2折扣、首相关MRR与空gold“不适用”。先算再运行。


In [ ]:
import math
def ranking_metrics(ranking,grades,k):
    if type(k) is not int or k<1 or len(ranking)!=len(set(ranking)):
        raise ValueError('positive k, unique IDs')
    if not grades or any(type(g) is not int or g<0 for g in grades.values()):raise ValueError('nonnegative integer grades')
    ideal=sorted(grades.values(),reverse=True)[:k]
    def dcg(values):return sum((2**g-1)/math.log2(i+2) for i,g in enumerate(values))
    denominator=dcg(ideal)
    if denominator==0:return {'mrr':None,'ndcg':None}
    actual=[grades.get(id,0) for id in ranking[:k]]
    mrr=next((1/(i+1) for i,g in enumerate(actual) if g>0),0.)
    return {'mrr':mrr,'ndcg':dcg(actual)/denominator}
grades={'a':3,'b':1,'c':0}
metrics=ranking_metrics(['b','a','c'],grades,3)
assert metrics['mrr']==1 and math.isclose(metrics['ndcg'],.7098097413968655,abs_tol=1e-12)
assert ranking_metrics(['a','b','c'],grades,3)['ndcg']==1
assert ranking_metrics(['x'],{'a':0},1)=={'mrr':None,'ndcg':None}
try:ranking_metrics(['a','a'],grades,2)
except ValueError:pass
else:raise AssertionError('duplicates inflate metric')
checks['graded_ranking']=True
print('graded metrics:',metrics)

## ANN与相关性Recall分开
只比较给定列表，不构建HNSW或运行embedding。


In [ ]:
def ann_recall(exact,approx,k):
    if type(k) is not int or k<1 or len(exact)<k or any(len(r)!=len(set(r)) for r in [exact,approx]):
        raise ValueError('exact k and unique ranks required')
    return len(set(exact[:k])&set(approx[:k]))/k
exact=['a','b'];approx=['a','c'];relevant={'c'}
assert ann_recall(exact,approx,2)==.5
relevance_recall=len(set(approx[:2])&relevant)/len(relevant)
assert relevance_recall==1 and not (set(exact)&relevant)
try:ann_recall(['a'],['a'],2)
except ValueError:pass
else:raise AssertionError('insufficient exact baseline accepted')
checks['ann_recall']=True
print('ANN Recall vs relevance Recall:',.5,relevance_recall)

## MaxSim逐query token聚合
固定相似度矩阵，非ColBERT编码器或benchmark。


In [ ]:
def maxsim(matrix):
    if not matrix or not matrix[0] or any(len(row)!=len(matrix[0]) for row in matrix):
        raise ValueError('nonempty rectangular matrix')
    if any(type(v) not in (int,float) or not math.isfinite(v) for row in matrix for v in row):
        raise ValueError('finite similarities')
    return sum(max(row) for row in matrix)
assert math.isclose(maxsim([[.9,.1],[.2,.8]]),1.7)
assert math.isclose(maxsim([[-.9,-.1],[-.2,-.8]]),-.3)
for invalid in [[],[[]],[[.9],[.2,.8]],[[math.nan]]]:
    try:maxsim(invalid)
    except ValueError:pass
    else:raise AssertionError('invalid similarity matrix accepted')
checks['late_interaction']=True
print('MaxSim:',maxsim([[.9,.1],[.2,.8]]))

## 记录与继续

在下一格 my_explanation 写下预测、改动与实际观察；运行后下载 mechanism_report.json，导回本篇笔记。下一次先独立解释边界，再对照旧记录。

In [ ]:
import json
from pathlib import Path
my_explanation = ''
report={'experiment':'lesson-retrieval-rag', 'scope':'stdlib fixtures; no model/service', 'checks':checks, 'my_explanation':my_explanation}
Path('mechanism_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))
